In [1]:
# /// script
# requires-python = ">=3.11,<3.14"
# dependencies = [
#     "azure-identity>=1.19.0",
#     "azure-mgmt-cognitiveservices>=13.5.0",
#     "azure-ai-projects>=2.0.0",
#     "openai>=1.75.0",
#     "python-dotenv>=1.0.0",
# ]
# ///
# ^ PEP 723 inline deps: juv/uv auto-manage this notebook's environment.
#   Run headless:  uvx juv run "Warner.AI-103.Lesson-02-runbook.ipynb"
#   VS Code venv:  uvx juv venv --from "Warner.AI-103.Lesson-02-runbook.ipynb"
print("Environment pinned inline. Kernel: AI-103 L2 Runbook (.venv)")

Environment pinned inline. Kernel: AI-103 L2 Runbook (.venv)


# Lesson 2 . Set up AI solutions in Foundry . Instructor copy

**Instructor twin of `m02_setup_demo.ipynb`.** Same four setup decisions, same live SDK code -- the code cells are **identical** to the student notebook and share the `LO# . Cell N` headings, so you can cross-match on screen.

This copy leads with a **PEP 723** header so `uvx juv run` builds the environment and picks the kernel with one command. Teaching markdown is grounded in current Microsoft Learn docs (links in each section).

| LO | Decision | Cross-match heading |
|---|---|---|
| **LO1** | design the infrastructure | `LO1 . Cell 1` |
| **LO2** | choose a deployment option | `LO2 + LO3 . Cell 2` |
| **LO3** | configure the deployment | `LO2 + LO3 . Cell 2` + `LO3 . Cell 3` |
| **LO4** | integrate with CI/CD | `LO4 . Cell 4` |

**Before recording:** `az login`, ensure `.env` sits next to this notebook (copy from `.env.example`).

## LO1 . Design the infrastructure -- two planes, one keyless identity

Setting up an AI solution in **Microsoft Foundry** starts with a decision most learners skip: *which Azure plane are you touching?* Azure separates every resource into a **control plane** and a **data plane**, and Foundry makes that split load-bearing.

| Plane | What you do here | SDK in this cell | Authorization surface |
|---|---|---|---|
| **Control plane** | create / configure / delete the deployment | `azure-mgmt-cognitiveservices` | Azure RBAC **actions** |
| **Data plane** | read and **call** the model | `azure-ai-projects` | Azure RBAC **dataActions** |

We build **one credential** and hand it to **both** clients. Auth is **keyless**: `AzureCliCredential` uses your `az login` token, so there is no application programming interface (API) key anywhere.

> **AI-103 exam tell:** **Owner** and **Contributor** have broad control-plane (ARM) permissions but **no data-plane dataActions**. To *build and call* in a Foundry project you need **Foundry User** (formerly **Azure AI User** -- you may still see the old name mid-rename). To *create deployments* you need a control-plane role such as **Cognitive Services Contributor** (or **Foundry Account Owner / Owner**). Two scopes, two roles.

*Grounding:* [Control plane vs data plane in Foundry](https://learn.microsoft.com/azure/foundry/concepts/authentication-authorization-foundry#control-plane-and-data-plane) . [Foundry RBAC built-in roles](https://learn.microsoft.com/azure/foundry/concepts/rbac-foundry#built-in-roles)

In [2]:
# LO1 . Cell 1: Two-plane clients (keyless Entra auth)
import os
from azure.identity import AzureCliCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient  # CONTROL plane
from azure.ai.projects import AIProjectClient                              # DATA plane
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")  # reads .env next to this notebook; copy from .env.example first


def required_env(name: str) -> str:
    """Return a required setting, or stop with the exact setup command."""
    value = os.environ.get(name)
    if value:
        return value
    raise RuntimeError(
        f"Missing {name}. Copy .env.example to .env next to this notebook, then fill it in.\n"
        "PowerShell: Copy-Item .env.example .env"
    )

# AzureCliCredential pins to the `az login` identity. DefaultAzureCredential checks
# EnvironmentCredential FIRST, so a stray AZURE_CLIENT_ID/SECRET could sign you in as a service
# principal instead. In CI/CD you swap this one line for DefaultAzureCredential (OIDC, still keyless).
cred = AzureCliCredential()                                    # one credential, two planes -- no keys
sub  = required_env("AZURE_SUBSCRIPTION_ID")
rg   = required_env("AZURE_RESOURCE_GROUP")
acct = required_env("FOUNDRY_ACCOUNT_NAME")
endpoint = required_env("FOUNDRY_PROJECT_ENDPOINT")

arm     = CognitiveServicesManagementClient(cred, sub)         # CONTROL plane: create/configure/delete
project = AIProjectClient(endpoint=endpoint, credential=cred)  # DATA plane: read/call
print("Control plane:", acct, "| Data plane:", endpoint)

Control plane: aif-ai103-core-quc6kxy2j2v2q | Data plane: https://aif-ai103-core-quc6kxy2j2v2q.services.ai.azure.com/api/projects/proj-lesson-02


## LO2 + LO3 . Choose a deployment option, then configure it

A **deployment** is a *named model endpoint your app calls by deployment name, not model name* -- itself a classic exam tell. Building the spec makes three decisions concrete.

**LO2 -- the deployment option (`sku.name`).** Microsoft's own guidance frames this as **two questions**: *where is data processed* and *what is my call volume?*

| `sku.name` | Billing | Best for |
|---|---|---|
| **`GlobalStandard`** | pay-per-token, highest quota | general workloads, the common starting point |
| `GlobalProvisionedManaged` | reserved **PTU** | predictable high throughput, low latency variance |
| `GlobalBatch` | **50% discount**, 24-hr async | large offline jobs |
| `DataZoneStandard` / `Standard` | pay-per-token | EU/US data-zone or single-region compliance |

**LO3 -- configure the deployment (two knobs the exam tests by name):**
- **`versionUpgradeOption`** pins the model version. `NoAutoUpgrade` = the deployment never auto-upgrades and **stops working when the version retires** (you point your code at a new version deliberately). The other options are `OnceCurrentVersionExpired` and `OnceNewDefaultVersionAvailable`. Careful: a **`null` policy behaves like `OnceCurrentVersionExpired`**, so "pinned" is a choice you make, not a default.
- **`raiPolicyName`** attaches the **content-filter (responsible-AI) policy** at the *deployment*, not the model. `Microsoft.DefaultV2` is the built-in default tier.

**LO4 seed:** `begin_create_or_update` is an **idempotent upsert** -- run it again and Azure converges to the same desired state. That single property is what makes it safe in a pipeline.

*Grounding:* [Deployment types](https://learn.microsoft.com/azure/foundry/foundry-models/concepts/deployment-types) . [Model version upgrade options](https://learn.microsoft.com/azure/foundry/openai/how-to/working-with-models#model-deployment-upgrade-configuration)

In [3]:
# LO2 + LO3 . Cell 2: Deployment spec -- type (SKU) + version pin + content filter
name = os.environ.get("DEPLOY_NAME", "ai103-l2-demo")

# gpt-5-nano is a cheap, current small model with quota in the eastus2 course account.
# The gpt-5 family REQUIRES an explicit version (no "service default" to fall back on) --
# which is itself the version-pinning lesson. Older gpt-4o-mini is blocked for NEW deployments.
model_block = {
    "format": os.environ.get("DEPLOY_MODEL_FORMAT", "OpenAI"),
    "name": os.environ.get("DEPLOY_MODEL_NAME", "gpt-5-nano"),
}
model_version = os.environ.get("DEPLOY_MODEL_VERSION", "2025-08-07")
if model_version:
    model_block["version"] = model_version

spec = {
    # LO2 -- the deployment OPTION. sku.name is elastic-vs-reserved-vs-batch in one string.
    "sku": {"name": os.environ.get("DEPLOY_SKU", "GlobalStandard"),
            "capacity": int(os.environ.get("DEPLOY_CAPACITY", "1"))},
    "properties": {
        "model": model_block,
        # LO3 -- CONFIGURE. NoAutoUpgrade pins the version; the deployment stops at retirement
        # rather than silently switching model weights under your app.
        "versionUpgradeOption": os.environ.get("DEPLOY_UPGRADE_OPTION", "NoAutoUpgrade"),
        # LO3 -- CONFIGURE. The content-filter (responsible-AI) policy is attached at the deployment.
        "raiPolicyName": os.environ.get("DEPLOY_RAI_POLICY", "Microsoft.DefaultV2"),
    },
}
# LO4 seed: begin_create_or_update is an idempotent upsert -- the same call a pipeline runs.
result = arm.deployments.begin_create_or_update(rg, acct, name, spec).result()
print("provisioningState:", result.properties.provisioning_state)

provisioningState: Succeeded


## LO3 . Verify the configuration stuck, then prove it is live

Two independent proofs, one per plane. First the **control plane** reads the deployment straight back -- the pinned version, the upgrade policy, and the content-filter policy are exactly what we sent. Then the **data plane** *calls* the same deployment through the OpenAI-compatible client and it answers. Portal and code agree because they were always the same resource.

> **Exam tell:** you call the model by its **deployment name** (`ai103-l2-demo`), never the underlying model id. That indirection is what lets you swap the model behind a stable endpoint.

*Grounding:* [Responses via the Foundry OpenAI client](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/deployment-types) (deployment-name-not-model-name)

In [4]:
# LO3 . Cell 3: Verify the config stuck, then call the deployment
live = arm.deployments.get(rg, acct, name)                     # read BACK from the control plane
print("sku.name            :", live.sku.name)
print("model.version       :", getattr(live.properties.model, "version", "(service default)"), " (pinned)")
print("versionUpgradeOption:", live.properties.version_upgrade_option, " (NoAutoUpgrade = pinned)")
print("raiPolicyName       :", live.properties.rai_policy_name)

# DATA plane proves the CONTROL plane: same deployment, now callable via the OpenAI-compatible client.
with project.get_openai_client() as client:
    r = client.responses.create(model=name, input="Reply with one word: ready.")
    print("live inference      :", r.output_text.strip())

sku.name            : GlobalStandard
model.version       : 2025-08-07  (pinned)
versionUpgradeOption: NoAutoUpgrade  (NoAutoUpgrade = pinned)
raiPolicyName       : Microsoft.DefaultV2


live inference      : ready


## LO4 . Integrate with CI/CD -- and tear down

The teardown is optional in real work; here we delete so the demo costs nothing to leave behind. The point that matters for the objective: because **create is an idempotent upsert** and auth is **secretless**, this whole notebook is pipeline-ready. In GitHub Actions you swap `AzureCliCredential` for `DefaultAzureCredential`, and `azure/login` supplies a **federated (OIDC) identity** -- no stored secret to leak. The spec dict maps 1:1 to a Bicep/ARM resource, so "treat AI infrastructure as code" is literal.

*Grounding:* [Control plane operations use ARM tooling (CLI, Bicep, Terraform)](https://learn.microsoft.com/azure/foundry/concepts/authentication-authorization-foundry#control-plane-and-data-plane)

In [5]:
# LO4 . Cell 4: Tear down so the meter stops (idempotent is the CI/CD story)
arm.deployments.begin_delete(rg, acct, name).result()
print("deleted:", name, "-- nothing left running.")

deleted: ai103-l2-demo -- nothing left running.


## Recap -- one deployment, four setup decisions

1. **LO1 infrastructure:** two planes, one keyless identity. Owner/Contributor are control-plane only; data actions need **Foundry User**.
2. **LO2 option:** `sku.name` picks billing + performance (`GlobalStandard` vs PTU vs Batch).
3. **LO3 configure:** `versionUpgradeOption` pins the version; `raiPolicyName` attaches the content filter -- both live on the deployment.
4. **LO4 CI/CD:** an idempotent, secretless upsert is pipeline-ready.

Every heading above (`LO# . Cell N`) matches the same code cell in the twin notebook, so the student and instructor copies line up one-for-one.